# Question-Answering LLM from scratch (TinyStories, Google Colab)

Everything here is trained from random initialisation - no pretrained weights, no pretrained tokenizer.

**Pipeline**
1. Load [TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories) from Hugging Face
2. Train our own byte-level BPE tokenizer (8k vocab)
3. Build a small GPT (~14M params) in plain PyTorch
4. **Stage 1 - pretrain** on stories (next-token prediction)
5. **Stage 2 - QA training**: TinyStories has no questions, so we auto-generate (story, question, answer) triples from *unseen* stories with regex templates, and train the model to answer (loss only on the answer tokens)
6. Evaluate and chat: paste a story + a question, get an answer

**Runtime:** Runtime > Change runtime type > **T4 GPU**. Total time is roughly 30-60 min; lower `PRETRAIN_ITERS` for a quick test.

In [ ]:
!pip -q install datasets tokenizers

In [ ]:
import os, math, random, re, time
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cpu":
    print("WARNING: no GPU. Switch the runtime to GPU (T4) or training will be extremely slow.")

torch.manual_seed(1337); random.seed(1337); np.random.seed(1337)

# ---- knobs ----
VOCAB_SIZE     = 8192
BLOCK_SIZE     = 512      # max context (story + question + answer must fit)
N_LAYER, N_HEAD, N_EMBD = 6, 6, 384
N_PRETRAIN     = 100000   # stories used for stage 1
N_QA_STORIES   = 60000    # separate stories used to make QA pairs
PRETRAIN_ITERS = 5000
PRETRAIN_BS    = 32
QA_ITERS       = 2000
QA_BS          = 32

device: cuda


## 1. Data

In [ ]:
from datasets import load_dataset

DS = "roneneldan/TinyStories"
train_raw = load_dataset(DS, split=f"train[:{N_PRETRAIN}]")
qa_raw    = load_dataset(DS, split=f"train[{N_PRETRAIN}:{N_PRETRAIN + N_QA_STORIES}]")
val_raw   = load_dataset(DS, split="validation[:2000]")

clean = lambda s: re.sub(r"\s+", " ", s).strip()
train_texts = [clean(x["text"]) for x in train_raw]
qa_texts    = [clean(x["text"]) for x in qa_raw]
val_texts   = [clean(x["text"]) for x in val_raw]
print(len(train_texts), len(qa_texts), len(val_texts))
print(train_texts[0][:400])

## 2. Tokenizer (trained from scratch)

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

tok = Tokenizer(models.BPE())
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tok.decoder = decoders.ByteLevel()
trainer = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=["<pad>", "<eos>"],
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
)
def batches(txts, n=1000):
    for i in range(0, len(txts), n):
        yield txts[i:i + n]
tok.train_from_iterator(batches(train_texts), trainer)
PAD, EOS = tok.token_to_id("<pad>"), tok.token_to_id("<eos>")
print("vocab:", tok.get_vocab_size(), "PAD:", PAD, "EOS:", EOS)

def encode_corpus(txts):
    ids = []
    for i in range(0, len(txts), 2000):
        for e in tok.encode_batch(txts[i:i + 2000]):
            ids.extend(e.ids); ids.append(EOS)
    return np.array(ids, dtype=np.uint16)

train_ids = encode_corpus(train_texts)
val_ids   = encode_corpus(val_texts)
print(f"train tokens: {len(train_ids)/1e6:.1f}M | val tokens: {len(val_ids)/1e6:.2f}M")

## 3. Model (GPT, from scratch)

In [ ]:
class Attention(nn.Module):
    def __init__(self):
        super().__init__()
        self.qkv  = nn.Linear(N_EMBD, 3 * N_EMBD, bias=False)
        self.proj = nn.Linear(N_EMBD, N_EMBD, bias=False)
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = [t.view(B, T, N_HEAD, C // N_HEAD).transpose(1, 2) for t in (q, k, v)]
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(y.transpose(1, 2).contiguous().view(B, T, C))

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(N_EMBD), nn.LayerNorm(N_EMBD)
        self.attn = Attention()
        self.mlp = nn.Sequential(nn.Linear(N_EMBD, 4 * N_EMBD), nn.GELU(), nn.Linear(4 * N_EMBD, N_EMBD))
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))

class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(VOCAB_SIZE, N_EMBD)
        self.pos_emb = nn.Embedding(BLOCK_SIZE, N_EMBD)
        self.blocks  = nn.ModuleList([Block() for _ in range(N_LAYER)])
        self.ln_f    = nn.LayerNorm(N_EMBD)
        self.head    = nn.Linear(N_EMBD, VOCAB_SIZE, bias=False)
        self.head.weight = self.tok_emb.weight          # weight tying
        self.apply(self._init)
    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        for b in self.blocks:
            x = b(x)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-100)
        return logits, loss

model = GPT().to(device)
print(f"params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

## 4. Stage 1 - pretrain on stories

In [ ]:
def get_batch(data, bs):
    ix = torch.randint(len(data) - BLOCK_SIZE - 1, (bs,))
    x = torch.stack([torch.from_numpy(data[i:i + BLOCK_SIZE].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i + 1:i + 1 + BLOCK_SIZE].astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)

def lr_at(it, total, max_lr, warmup=200):
    if it < warmup: return max_lr * (it + 1) / warmup
    p = (it - warmup) / max(1, total - warmup)
    return max_lr * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * p)))

use_amp = device == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
def amp_ctx(): return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp)

@torch.no_grad()
def eval_loss(data, n=30, bs=16):
    model.eval(); ls = []
    for _ in range(n):
        x, y = get_batch(data, bs)
        with amp_ctx(): _, l = model(x, y)
        ls.append(l.item())
    model.train(); return float(np.mean(ls))

def train_step(x, y, lr, opt):
    for g in opt.param_groups: g["lr"] = lr
    with amp_ctx(): _, loss = model(x, y)
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(opt); scaler.update()
    return loss.item()

opt = torch.optim.AdamW(model.parameters(), lr=1e-3, betas=(0.9, 0.95), weight_decay=0.1)
model.train(); t0 = time.time()
for it in range(PRETRAIN_ITERS):
    x, y = get_batch(train_ids, PRETRAIN_BS)
    loss = train_step(x, y, lr_at(it, PRETRAIN_ITERS, 1e-3), opt)
    if it % 100 == 0:
        print(f"iter {it:5d} | loss {loss:.3f} | {time.time()-t0:.0f}s")
    if it % 1000 == 0 or it == PRETRAIN_ITERS - 1:
        print(f"   >> val loss {eval_loss(val_ids):.3f}")
torch.save(model.state_dict(), "pretrained.pt")

In [ ]:
@torch.no_grad()
def generate(prompt_ids, max_new=200, temperature=0.8, top_k=40, stop_at_eos=True):
    model.eval()
    idx = torch.tensor([prompt_ids], device=device)
    out = []
    for _ in range(max_new):
        with amp_ctx():
            logits, _ = model(idx[:, -BLOCK_SIZE:])
        logits = logits[:, -1, :].float()
        if temperature == 0:
            nxt = logits.argmax(-1, keepdim=True)
        else:
            logits = logits / temperature
            if top_k:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float("inf")
            nxt = torch.multinomial(F.softmax(logits, -1), 1)
        if stop_at_eos and nxt.item() == EOS: break
        idx = torch.cat([idx, nxt], 1); out.append(nxt.item())
    return tok.decode(out)

print(generate(tok.encode("Once upon a time, there was a little girl named").ids))

## 5. Build QA data from unseen stories
TinyStories has no questions, so we derive them with simple templates. Because the answers come straight from each story, the model must learn to **read the story and answer from it** (not just memorise).

In [ ]:
FEELS = "happy|sad|scared|excited|proud|angry|surprised|tired|lonely|nervous|brave|curious|worried|sorry"
KINDS = "girl|boy|cat|dog|bunny|bird|rabbit|bear|mouse|fox|duck|robot|dragon|frog|monkey|elephant|lion|turtle"

def make_qa(story):
    m = re.search(r"named ([A-Z][a-z]+)", story)
    if not m: return []
    n = re.escape(m.group(1)); name = m.group(1)
    qa = [("What is the name of the main character?", name)]
    k = re.search(rf"(?:a|an) (?:little |young |small |big )?({KINDS}) named {n}", story)
    if k: qa.append((f"What kind of character is {name}?", k.group(1)))
    f = re.search(rf"{n} (?:was|felt|got|became) (?:very |so |really )?({FEELS})", story)
    if f: qa.append((f"How did {name} feel?", f.group(1)))
    l = re.search(rf"{n} (?:loved|liked) to ([^.,!?]+)[.,!?]", story)
    if l: qa.append((f"What did {name} like to do?", l.group(1)))
    d = re.search(rf"{n} found (?:a|an|the) ([^.,!?]+?)(?: in| on| under| near| at| and|[.,!?])", story)
    if d: qa.append((f"What did {name} find?", d.group(1)))
    return qa

def prompt_text(story, q): return f"Story: {story}\nQuestion: {q}\nAnswer:"

def build_examples(stories, cap=None):
    ex = []
    for s in stories:
        for q, a in make_qa(s):
            p = tok.encode(prompt_text(s, q)).ids
            ans = tok.encode(" " + a).ids + [EOS]
            if len(p) + len(ans) <= BLOCK_SIZE:
                ex.append((p, ans))
    random.shuffle(ex)
    return ex[:cap] if cap else ex

qa_train = build_examples(qa_texts, cap=150000)
qa_val   = build_examples(val_texts, cap=500)
print("QA train examples:", len(qa_train), "| val examples:", len(qa_val))
p, a = qa_train[0]; print(tok.decode(p)[-200:], "->", tok.decode(a))

## 6. Stage 2 - train the model to answer (loss on answer tokens only)

In [ ]:
def qa_batch(examples, bs):
    batch = random.sample(examples, bs)
    L = max(len(p) + len(a) for p, a in batch)
    x = torch.full((bs, L), PAD, dtype=torch.long)
    y = torch.full((bs, L), -100, dtype=torch.long)
    for i, (p, a) in enumerate(batch):
        seq = p + a
        x[i, :len(seq)] = torch.tensor(seq)
        # predict token t+1 from t; supervise only the answer part
        y[i, len(p) - 1:len(seq) - 1] = torch.tensor(a)
    return x.to(device), y.to(device)

model.load_state_dict(torch.load("pretrained.pt"))
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, betas=(0.9, 0.95), weight_decay=0.01)
model.train(); t0 = time.time()
for it in range(QA_ITERS):
    x, y = qa_batch(qa_train, QA_BS)
    loss = train_step(x, y, lr_at(it, QA_ITERS, 3e-4, warmup=100), opt)
    if it % 100 == 0:
        print(f"iter {it:5d} | loss {loss:.3f} | {time.time()-t0:.0f}s")
torch.save(model.state_dict(), "qa_model.pt")
tok.save("tokenizer.json")

## 7. Evaluate on held-out stories

In [ ]:
def answer(story, question, max_new=16):
    ids = tok.encode(prompt_text(clean(story), question.strip())).ids
    ids = ids[-(BLOCK_SIZE - max_new):]
    return generate(ids, max_new=max_new, temperature=0, top_k=None).strip()

hits = 0
for p, a in qa_val:
    out = generate(p, max_new=16, temperature=0, top_k=None).strip().lower()
    gold = tok.decode(a).strip().lower()
    hits += out == gold
print(f"exact match on {len(qa_val)} held-out questions: {hits/len(qa_val):.1%}")

## 8. Try it

In [ ]:
story = """Once upon a time, there was a little girl named Mia. Mia loved to play with her red ball in the park.
One sunny day, she found a shiny key under a big tree. Mia was very excited. She ran home to show her mom."""

for q in ["What is the name of the main character?",
          "What did Mia like to do?",
          "What did Mia find?",
          "How did Mia feel?"]:
    print(f"Q: {q}\nA: {answer(story, q)}\n")

## 9. (Optional) Save to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/tinystories_qa", exist_ok=True)
!cp pretrained.pt qa_model.pt tokenizer.json /content/drive/MyDrive/tinystories_qa/

## Where to go next
- **Scale up:** more stories (`N_PRETRAIN` up to ~2M), `N_LAYER=8`, `N_EMBD=512`, more iterations.
- **Richer questions:** add more templates to `make_qa`, or mix in a real QA dataset (e.g. SQuAD) in stage 2; a bigger model and far more data are needed for that.
- **Honest limit:** a model this small can answer the question types it was trained on about TinyStories-style text. It is not an open-domain assistant.